# SenChronoAge

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.SenChronoAge)

class SenChronoAge(LinearReferenceClock):
    pass



In [3]:
model = pya.models.SenChronoAge()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = 'senchronoage'
model.metadata["data_type"] = 'methylation'
model.metadata["species"] = 'Homo sapiens'
model.metadata["year"] = 2026
model.metadata["approved_by_author"] = '⌛'
model.metadata["citation"] = "Kasamoto, Kotaro, et al. \"DNA methylation clocks for estimating replicative senescence in human cells.\" Aging Cell (2026): e70430."
model.metadata["doi"] = "https://doi.org/10.1111/acel.70430"
model.metadata["research_only"] = None
model.metadata["notes"] = None

## Download clock dependencies

In [5]:
os.system(f"curl -sL -o SenChronoAge_CpGs.csv https://raw.githubusercontent.com/HigginsChenLab/methylCIPHER/19b12296b0d7eb7055a97d068064df635f44ce3e/data-raw/SenescenceAge/SenChronoAge_CpGs.csv")

0

## Load features

In [6]:
coef_df = pd.read_csv('SenChronoAge_CpGs.csv')
model.features = coef_df['CpG'].tolist()

## Load weights into base model

In [7]:
weights = torch.tensor(coef_df['Coefficient'].tolist()).unsqueeze(0).float()
intercept = torch.tensor([-80.0532]).float()

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Kasamoto, Kotaro, et al. "DNA methylation clocks for estimating '
             'replicative senescence in human cells." Aging Cell (2026): '
             'e70430.',
 'clock_name': 'senchronoage',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1111/acel.70430',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2026}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg00210473', 'cg00254017', 'cg00296652', 'cg00602326', 'cg00922748', 'cg00964103', 'cg01013171', 'cg01079407', 'cg01141812', 'cg01196788', 'cg01249187', 'cg01263575', 'cg01302656', 'cg01654770', 'cg01675158', 'cg01769968', 'cg01987776', 'cg02716556', 'cg02796545', 'cg03649284', 'cg03660134', 'cg04094193', 'cg

## Basic test

In [13]:
torch.manual_seed(42)
input = torch.randn(10, len(model.features), dtype=float)
model.eval()
model.to(float)
pred = model(input)
pred

tensor([[ -54.5464],
        [-218.2588],
        [ 143.5234],
        [-101.8496],
        [-296.5361],
        [-180.5059],
        [-193.5961],
        [-180.5310],
        [ -16.9680],
        [  65.8968]], dtype=torch.float64, grad_fn=<AddmmBackward0>)

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: SenChronoAge_CpGs.csv
